# 14 · async / await Basics

After this notebook you can explain the event loop in plain English, run LLM calls concurrently with `gather`, cap them with a `Semaphore`, bound them with timeouts, stream tokens with `async for`, cancel work cleanly — and say when async does **not** help (CPU-bound work).

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [12 · Decorators and context managers](12_decorators_and_context_managers.ipynb) (`async with` builds on `with`), [09 · Errors and exceptions](09_errors_and_exceptions.ipynb) (`ExceptionGroup`)

## Why this matters in interviews

- An LLM call is mostly waiting on the network. "Why would you use async for an endpoint that calls an LLM?" is a standard question, and the follow-up is always "what happens if you call something blocking inside it?"
- Agents fan out — several searches, several tool calls, several models — and interviewers want to hear *concurrency cap, timeout, and what happens on partial failure*.
- Streaming, time-to-first-token and cancellation are everyday production topics; all three are async generators and tasks underneath.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tl08` | Why would you use async for an endpoint that calls an LLM? |
| `ag32` | How do you parallelise agent work safely? |
| `in29` | What is the difference between concurrency, batch size and throughput? (the concurrency part) |
| `in26` | How should you handle an LLM timeout? |
| `po17` | Why are timeouts more delicate in an LLM system than in a normal service? |
| `pd39` | What does "streaming" mean in a production LLM system, and what are the gotchas? |

The cell below finds `llm_setup.py` (the model is only used in section 12; with no API key and no Ollama it uses a deterministic offline stand-in).

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

## 1. Why async: most of an LLM call is waiting

**In plain English:** one waiter can serve ten tables, because most of each table's time is spent reading the menu or eating — the waiter takes an order, walks away, and comes back when someone is ready. Blocking code is a waiter who stands at one table until it has finished dessert.

An LLM request is ~99% waiting for someone else's GPU. With `async`, one Python thread starts a request, and while that request waits it switches to another:

```
sequential   [ call 1 ======== ][ call 2 ======== ][ call 3 ======== ]     3 x 0.3 s = 0.9 s
concurrent   [ call 1 ======== ]
             [ call 2 ======== ]                                           about 0.3 s
             [ call 3 ======== ]
             ===== = waiting on the network; the CPU is idle
```

| Term | Meaning | Python tool |
|---|---|---|
| **Concurrency** | many tasks *in progress* at once, taking turns on one CPU | `asyncio`, threads |
| **Parallelism** | many tasks *executing* at the same instant on several CPUs | `multiprocessing`, free-threaded Python |

Async gives you concurrency: perfect for I/O (HTTP, databases, LLM APIs), useless for number crunching.

## 2. Coroutine functions vs normal functions

`async def` defines a **coroutine function**. Calling it does not run anything — it returns a **coroutine object**, a paused piece of work. `await` runs it, and while it is waiting (`await asyncio.sleep`, `await client.post`) the event loop runs other coroutines.

**Predict:** what does `fetch(1)` return, and how long does that line take?

In [ ]:
import asyncio
import inspect
import time

async def fetch(i):
    await asyncio.sleep(0.1)            # a non-blocking wait: the loop may run other tasks meanwhile
    return f"doc {i}"

start = time.perf_counter()
coro = fetch(1)                         # nothing has run yet
print("fetch(1) returned a", type(coro).__name__, f"after {(time.perf_counter() - start) * 1000:.2f} ms")
print("is a coroutine function?", inspect.iscoroutinefunction(fetch))
result = await coro                     # top-level await: allowed in Jupyter (see section 3)
print("await gave:", repr(result), f"after {(time.perf_counter() - start) * 1000:.0f} ms")

forgot = fetch(2)                       # the classic bug: calling without await
print("forgot to await ->", forgot)
forgot.close()                          # (tidy up so Python does not warn "coroutine was never awaited")
assert result == "doc 1"

## 3. The event loop — and why Jupyter is different from a script

The **event loop** is the scheduler: it keeps a list of tasks, runs one until it hits an `await` on something not ready yet, then switches to another task that *is* ready. Nothing runs in parallel; tasks politely take turns at every `await`.

| | Script or service | Jupyter / IPython |
|---|---|---|
| Who runs the loop | you: `asyncio.run(main())`, once, at the entry point | the kernel already has a loop running |
| Top-level `await` | a `SyntaxError` outside `async def` | allowed in any cell |
| `asyncio.run(...)` | the right way in | `RuntimeError`, as shown below |
| Web servers (FastAPI, uvicorn) | the server owns the loop; you write `async def` endpoints | — |

A script version of this notebook would look like:

```python
import asyncio

async def main():
    answers = await asyncio.gather(*(ask(q) for q in questions))
    print(answers)

if __name__ == "__main__":
    asyncio.run(main())
```

In [ ]:
loop = asyncio.get_running_loop()
print("this kernel already has a running loop:", loop.is_running())

pending = fetch(3)
try:
    asyncio.run(pending)
except RuntimeError as e:
    print("asyncio.run() in Jupyter ->", e)
    pending.close()

## 4. `gather`, `create_task`, and partial failure

`asyncio.gather(a, b, c)` runs the coroutines **concurrently** and returns their results **in argument order** — not in the order they finish. `asyncio.create_task(coro)` schedules a coroutine to start running at the next `await`, so you can start work, do something else, and collect it later.

**Predict:** in which order do the three results come back?

In [ ]:
async def fake_llm(prompt, delay):
    await asyncio.sleep(delay)
    return f"{prompt} ({delay}s)"

start = time.perf_counter()
one = await fake_llm("q1", 0.2)
two = await fake_llm("q2", 0.2)
three = await fake_llm("q3", 0.2)
t_sequential = time.perf_counter() - start

start = time.perf_counter()
results = await asyncio.gather(fake_llm("slow", 0.3), fake_llm("fast", 0.1), fake_llm("medium", 0.2))
t_gather = time.perf_counter() - start
print(f"sequential: {t_sequential:.2f} s | gather: {t_gather:.2f} s -> {results}")

task = asyncio.create_task(fake_llm("background summary", 0.2))    # starts at the next await
await asyncio.sleep(0.1)                                            # ...do other work meanwhile
print("task done after 0.1 s?", task.done(), "| result:", await task)
assert results == ["slow (0.3s)", "fast (0.1s)", "medium (0.2s)"]
assert t_sequential > 0.55 and t_gather < 0.45

**Partial failure** is the interesting case (ag32): three tool calls, one times out. Two tools with different semantics:

- `gather(..., return_exceptions=True)` returns exceptions **as values** next to the successes, so you can continue with what you have — and tell the model which branch is missing. (Without it, the first exception propagates, but the other tasks keep running in the background.)
- `asyncio.TaskGroup` (3.11+, "structured concurrency") treats the group as one unit: the first failure **cancels the siblings** and raises an `ExceptionGroup`, which you handle with `except*`.

In [ ]:
async def tool_call(name, delay, fail=False):
    await asyncio.sleep(delay)
    if fail:
        raise TimeoutError(f"{name} timed out")
    return f"{name}: ok"

outcomes = await asyncio.gather(tool_call("search", 0.1), tool_call("weather", 0.05, fail=True),
                                tool_call("calendar", 0.1), return_exceptions=True)
print("gather(return_exceptions=True):", outcomes)
succeeded = [o for o in outcomes if not isinstance(o, BaseException)]
missing = [str(o) for o in outcomes if isinstance(o, BaseException)]
print(f"continue with {len(succeeded)} results and tell the model: {missing}")

try:
    async with asyncio.TaskGroup() as tg:
        tasks = [tg.create_task(tool_call("search", 0.1)), tg.create_task(tool_call("weather", 0.05, fail=True)),
                 tg.create_task(tool_call("calendar", 0.1))]
except* TimeoutError as group:
    print("TaskGroup raised an ExceptionGroup:", [str(e) for e in group.exceptions])
print("sibling tasks:", ["cancelled" if t.cancelled() else "finished" for t in tasks])
assert len(succeeded) == 2 and tasks[0].cancelled() and tasks[2].cancelled()

## 5. `Semaphore` — cap the concurrency

Firing 500 requests at once gets you 429s from the provider and exhausts your own connection pool. A `Semaphore(3)` is a bouncer with three wristbands: `async with sem:` waits until one is free. The timeline below shows ten 0.2-second calls running in waves of at most three.

In [ ]:
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e1e0d9", "axes.axisbelow": True})

sem = asyncio.Semaphore(3)
timeline, in_flight, peak = [], 0, 0
t0 = time.perf_counter()

async def limited_call(i):
    global in_flight, peak
    async with sem:                                   # at most 3 coroutines inside this block
        in_flight += 1
        peak = max(peak, in_flight)
        started = time.perf_counter() - t0
        await asyncio.sleep(0.2)
        in_flight -= 1
        timeline.append((i, started, time.perf_counter() - t0))

await asyncio.gather(*(limited_call(i) for i in range(10)))
total = max(end for _, _, end in timeline)
print(f"peak in flight: {peak} | total: {total:.2f} s for 10 calls of 0.2 s")

fig, ax = plt.subplots(figsize=(9, 3.6))
for i, started, ended in timeline:
    ax.barh(i, ended - started, left=started, height=0.6, color=BLUE)
ax.set_yticks(range(10), [f"call {i}" for i in range(10)]); ax.invert_yaxis()
ax.set_xlabel("seconds since start"); ax.grid(axis="y", visible=False)
ax.set_title("asyncio.Semaphore(3): ten 0.2 s calls run in waves of three")
plt.show()
assert peak == 3 and 0.75 < total < 1.2

This is the concurrency knob from `in29`: raising it raises throughput until you hit the provider's rate limit or the server's capacity, after which extra concurrency only adds queueing and 429s. Set it from your quota, not from optimism.

## 6. Timeouts: `asyncio.timeout` and `wait_for`

Never wait forever — library defaults are often very long or unlimited. Two spellings:

- `async with asyncio.timeout(seconds):` (3.11+) — bounds a whole block;
- `await asyncio.wait_for(coro, timeout=seconds)` — bounds one awaitable.

Both cancel the work and raise `TimeoutError`. The `in26` pattern is: a timeout shorter than the user's patience, then a fallback that is **marked as degraded** (and never cached).

In [ ]:
async def slow_model():
    await asyncio.sleep(1.0)
    return "a thorough answer from the big model"

async def small_model():
    await asyncio.sleep(0.05)
    return "a shorter answer from a smaller model"

try:
    async with asyncio.timeout(0.2):
        await slow_model()
except TimeoutError:
    print("asyncio.timeout(0.2): gave up on the slow model")

async def answer_with_fallback(budget_s):
    try:
        return await asyncio.wait_for(slow_model(), timeout=budget_s), False
    except TimeoutError:
        return await small_model(), True             # degraded: label it, don't cache it

start = time.perf_counter()
answer, degraded = await answer_with_fallback(budget_s=0.2)
print(f"answer={answer!r} degraded={degraded} after {time.perf_counter() - start:.2f} s")
assert degraded and time.perf_counter() - start < 0.5

`po17`'s senior point is **deadline propagation**: a request has one budget, and every step must fit inside what is left of it. A 20-second timeout on each of three steps inside a 10-second request budget is a timeout that never fires usefully. One `asyncio.timeout` around the whole chain enforces the real budget:

In [ ]:
completed = []

async def step(name, seconds):
    await asyncio.sleep(seconds)
    completed.append(name)

try:
    async with asyncio.timeout(0.5):                 # ONE budget for the whole request
        await step("retrieve", 0.15)
        await step("rerank", 0.15)
        await step("generate", 0.4)                  # would blow the budget
except TimeoutError:
    print("request deadline hit; finished steps:", completed)
assert completed == ["retrieve", "rerank"]

## 7. `as_completed` and `wait` — act on results as they finish

`gather` waits for everything. `asyncio.as_completed` hands you results **fastest first** — useful to show partial results, or to take the first good answer. `asyncio.wait(..., return_when=FIRST_COMPLETED)` is the building block for **hedging** (in27): send the same request to two backends, keep the first reply, cancel the other.

In [ ]:
async def ask_backend(name, delay):
    await asyncio.sleep(delay)
    return name

finish_order = []
for next_done in asyncio.as_completed([ask_backend("gpt-4.1-mini", 0.3), ask_backend("gpt-4.1-nano", 0.1),
                                       ask_backend("local-3b", 0.2)]):
    finish_order.append(await next_done)
print("as_completed order:", finish_order)

hedges = [asyncio.create_task(ask_backend("region-eu", 0.25)), asyncio.create_task(ask_backend("region-us", 0.1))]
done, pending = await asyncio.wait(hedges, return_when=asyncio.FIRST_COMPLETED)
for t in pending:
    t.cancel()                                        # don't pay for (or wait on) the loser
print("hedged request answered by:", done.pop().result(), "| cancelled:", len(pending))
assert finish_order == ["gpt-4.1-nano", "local-3b", "gpt-4.1-mini"]

## 8. Async generators and `async for` — streaming

An `async def` containing `yield` is an **async generator**: it produces items over time, and you consume it with `async for` (or an async comprehension). This is exactly the shape of a streaming LLM response. Streaming does not make the answer finish sooner — it makes the **first token** arrive sooner, which is what the user feels (pd39).

In [ ]:
async def stream_tokens(text, delay=0.03):
    for token in text.split():
        await asyncio.sleep(delay)                   # each token arrives after a short wait
        yield token + " "

start, first_token_at, received = time.perf_counter(), None, []
async for token in stream_tokens("Refunds are issued within 30 days of delivery, excluding digital goods."):
    if first_token_at is None:
        first_token_at = time.perf_counter() - start
    received.append(token)
total_s = time.perf_counter() - start
print(f"time to first token: {first_token_at * 1000:.0f} ms | full answer: {total_s * 1000:.0f} ms")
print("".join(received))

words = [t.strip() async for t in stream_tokens("async comprehensions work too", delay=0.01)]
print("async comprehension:", words)
assert first_token_at < total_s / 4

## 9. Cancellation

`task.cancel()` raises `asyncio.CancelledError` inside the task at its current `await`. That is how timeouts work, and how you stop a generation when the user closes the tab. Rules: do cleanup in `finally`, and if you catch `CancelledError`, **re-raise it** — swallowing it makes the task impossible to stop. (It inherits from `BaseException`, so `except Exception` does not catch it by accident.)

In [ ]:
cleanup_log = []

async def long_generation():
    try:
        await asyncio.sleep(10)                       # a long stream the user abandons
    except asyncio.CancelledError:
        cleanup_log.append("cancelled: stop reading the stream")
        raise                                         # always re-raise
    finally:
        cleanup_log.append("finally: connection released")

task = asyncio.create_task(long_generation())
await asyncio.sleep(0.05)
task.cancel()
try:
    await task
except asyncio.CancelledError:
    print("the code awaiting the task sees CancelledError as well")
print("task.cancelled():", task.cancelled(), "| cleanup:", cleanup_log)
assert task.cancelled() and cleanup_log == ["cancelled: stop reading the stream", "finally: connection released"]

## 10. The number-one async bug: blocking the event loop

Inside `async def`, anything that blocks — `time.sleep`, the sync `requests` or `openai.OpenAI` client, a sync database driver, a heavy JSON parse — **freezes the whole loop**, so every other request waits too (the tl08 trap). Use async libraries (`httpx.AsyncClient`, `AsyncOpenAI`), or push unavoidable blocking calls to a thread with `asyncio.to_thread`.

**Predict:** how long do three "concurrent" calls take when each does `time.sleep(0.2)`?

In [ ]:
async def blocking_call(i):
    time.sleep(0.2)                                  # SYNC sleep: blocks the entire loop
    return i

async def offloaded_call(i):
    await asyncio.to_thread(time.sleep, 0.2)          # runs in a worker thread; the loop stays free
    return i

start = time.perf_counter()
await asyncio.gather(*(blocking_call(i) for i in range(3)))
t_blocked = time.perf_counter() - start
start = time.perf_counter()
await asyncio.gather(*(offloaded_call(i) for i in range(3)))
t_offloaded = time.perf_counter() - start
print(f"time.sleep inside async def: {t_blocked:.2f} s (no concurrency at all)")
print(f"asyncio.to_thread          : {t_offloaded:.2f} s")
assert t_blocked > 0.55 and t_offloaded < 0.45

## 11. Sequential vs threads vs asyncio — and the CPU-bound caveat

Ten simulated I/O calls of 0.1 s each, three ways. Then four pure-Python CPU-heavy tasks, the same three ways.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def io_task(_):
    time.sleep(0.1)                                   # stands in for a blocking network call

async def io_task_async():
    await asyncio.sleep(0.1)

def cpu_task(_=None):
    return sum(i * i for i in range(400_000))         # pure Python arithmetic: holds the GIL

async def cpu_task_async():
    return cpu_task()                                 # no await inside: async cannot help

def seconds(fn, repeat=3):                            # best of 3 runs: less noise from other programs
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter(); fn(); best = min(best, time.perf_counter() - start)
    return best

async def seconds_async(make_coros, repeat=3):
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter(); await asyncio.gather(*make_coros()); best = min(best, time.perf_counter() - start)
    return best

with ThreadPoolExecutor(max_workers=10) as pool:
    io = {"sequential": seconds(lambda: [io_task(i) for i in range(10)]),
          "threads": seconds(lambda: list(pool.map(io_task, range(10)))),
          "asyncio": await seconds_async(lambda: [io_task_async() for _ in range(10)])}
with ThreadPoolExecutor(max_workers=4) as pool:
    cpu = {"sequential": seconds(lambda: [cpu_task() for _ in range(4)]),
           "threads": seconds(lambda: list(pool.map(cpu_task, range(4)))),
           "asyncio": await seconds_async(lambda: [cpu_task_async() for _ in range(4)])}
for kind, res in [("I/O-bound", io), ("CPU-bound", cpu)]:
    print(f"{kind}: " + " | ".join(f"{k} {v:.2f} s" for k, v in res.items()))
assert io["threads"] < io["sequential"] / 3 and io["asyncio"] < io["sequential"] / 3
assert cpu["threads"] > 0.6 * cpu["sequential"]            # the GIL: threads don't speed up pure-Python CPU work

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
colors = {"sequential": ORANGE, "threads": AQUA, "asyncio": BLUE}
for ax, (title, res) in zip(axes, [("I/O-bound: 10 waits of 0.1 s", io), ("CPU-bound: 4 pure-Python loops", cpu)]):
    bars = ax.bar(list(res), list(res.values()), color=[colors[k] for k in res], width=0.6)
    ax.bar_label(bars, fmt="%.2f s", padding=2, fontsize=9)
    ax.set_title(title); ax.set_ylabel("wall-clock seconds"); ax.grid(axis="x", visible=False)
fig.suptitle("Threads and asyncio overlap waiting; neither speeds up Python number crunching")
plt.tight_layout()
plt.show()

Why the right-hand panel looks like that: CPython's **GIL** (global interpreter lock) lets only one thread execute Python bytecode at a time, so threads interleave CPU work instead of running it in parallel; asyncio runs everything on one thread anyway. For CPU-bound work — tokenising a large corpus, parsing PDFs, pure-Python scoring — use **processes**, which each have their own interpreter and GIL:

```python
from concurrent.futures import ProcessPoolExecutor

def heavy(chunk): ...                      # must live in an importable .py module, not a notebook cell,
                                           # because Windows/macOS start workers by importing it
if __name__ == "__main__":                 # the guard from notebook 11: workers re-import this file
    with ProcessPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(heavy, chunks))
```

From async code you can do `await loop.run_in_executor(process_pool, heavy, chunk)`. Also worth one sentence in an interview: NumPy, PyTorch and tokenizers release the GIL inside their C/Rust code, so threads *can* help there; Python 3.13 added an experimental **free-threaded** build without the GIL (PEP 703), made officially supported in 3.14 — check with `sys._is_gil_enabled()`.

## 12. Practical: 10 concurrent LLM calls

Now the real thing, through the async client from `llm_setup`. Offline, the stand-in is told to take 0.3 s per request (`OFFLINE.latency`), roughly like a fast hosted model. Three strategies for the same 10 questions: one after another, all at once with `gather`, and `gather` capped at three in flight.

In [ ]:
from llm_setup import make_async_client, OFFLINE

if OFFLINE:
    OFFLINE.latency = 0.3                          # simulate network + model time per request

aclient = make_async_client()                      # an openai.AsyncOpenAI for the active provider
questions = [f"What is {n} * 7?" for n in range(1, 11)]

async def ask(question):
    resp = await aclient.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": question}], max_tokens=20)
    return resp.choices[0].message.content

await ask("What is 1 + 1?")                        # warm-up: the first request pays one-off setup costs
print("provider:", PROVIDER, "| model:", MODEL)

In [ ]:
start = time.perf_counter()
sequential_answers = [await ask(q) for q in questions]              # one at a time
t_seq = time.perf_counter() - start

start = time.perf_counter()
gather_answers = await asyncio.gather(*(ask(q) for q in questions))  # all ten at once
t_all = time.perf_counter() - start

limit = asyncio.Semaphore(3)

async def ask_limited(question):
    async with limit:
        return await ask(question)

start = time.perf_counter()
limited_answers = await asyncio.gather(*(ask_limited(q) for q in questions))   # at most 3 in flight
t_sem = time.perf_counter() - start

print(f"sequential {t_seq:.2f} s | gather {t_all:.2f} s | gather + Semaphore(3) {t_sem:.2f} s")
print("first answers:", gather_answers[:3])
assert len(sequential_answers) == len(gather_answers) == len(limited_answers) == 10
if OFFLINE:                                        # the stand-in's latency is exact, so check the claim precisely
    assert t_all < t_seq / 3 and t_all < t_sem < t_seq

In [ ]:
timings = {"sequential\n(for loop)": t_seq, "gather\n(all 10 at once)": t_all, "gather +\nSemaphore(3)": t_sem}
fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.bar(list(timings), list(timings.values()), color=BLUE, width=0.55)
ax.bar_label(bars, fmt="%.2f s", padding=2)
ax.set_ylabel("wall-clock seconds for 10 calls"); ax.grid(axis="x", visible=False)
ax.set_title(f"10 chat completions via AsyncOpenAI ({PROVIDER})")
plt.show()

Offline, the numbers follow directly from the 0.3 s latency: about 10 × 0.3 s sequentially, about one latency with `gather`, and about four "waves" with `Semaphore(3)`. Against a real API you will see more spread (real latency varies per call), and a local Ollama server may process requests one at a time, in which case `gather` cannot beat the server's own queue.

Next, **streaming** with the same client: `stream=True` turns the response into an async iterator of chunks. Time to first token (TTFT) is what users perceive; total time is unchanged.

In [ ]:
if OFFLINE:
    OFFLINE.token_delay = 0.02                     # gap between streamed chunks

start, ttft, pieces = time.perf_counter(), None, []
stream = await aclient.chat.completions.create(
    model=MODEL, stream=True, max_tokens=60,
    messages=[{"role": "user", "content": "Summarise in one sentence: Async lets one Python process wait on "
               "many network calls at once. The event loop switches tasks whenever one is waiting."}])
async for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        if ttft is None:
            ttft = time.perf_counter() - start
        pieces.append(chunk.choices[0].delta.content)
total_stream = time.perf_counter() - start
print(f"TTFT {ttft * 1000:.0f} ms | total {total_stream * 1000:.0f} ms | {len(pieces)} chunks")
print("".join(pieces))
assert ttft is not None and ttft < total_stream

Finally, a **timeout with a labelled fallback** around a real call. The 0.1 s budget is deliberately shorter than the request, so the fallback path runs; in production the budget comes from your latency SLO.

In [ ]:
try:
    reply, degraded = await asyncio.wait_for(ask("What is 12 * 12?"), timeout=0.1), False
except TimeoutError:
    reply, degraded = "The assistant is slow right now - here is a cached FAQ link instead.", True
print(f"degraded={degraded}: {reply}")
if OFFLINE:
    assert degraded                                # 0.3 s simulated latency > 0.1 s budget

if OFFLINE:                                        # put the stand-in back to instant replies
    OFFLINE.latency = 0
    OFFLINE.token_delay = 0
await aclient.close()

## Try it yourself

**1.** Write `gather_with_limit(coros, limit)` that runs any list of coroutines with at most `limit` in flight and returns results in input order. Test it with six `asyncio.sleep(0.1)` calls and `limit=2` (expect about 0.3 s).

In [ ]:
# Solution — try it yourself first, then run this
async def gather_with_limit(coros, limit):
    sem = asyncio.Semaphore(limit)

    async def run(coro):
        async with sem:
            return await coro

    return await asyncio.gather(*(run(c) for c in coros))

async def sleepy(i):
    await asyncio.sleep(0.1)
    return i

start = time.perf_counter()
out = await gather_with_limit([sleepy(i) for i in range(6)], limit=2)
elapsed = time.perf_counter() - start
print(out, f"{elapsed:.2f} s")
assert out == list(range(6)) and 0.25 < elapsed < 0.5

**2.** Give three sequential 0.1 s steps a **total** budget of 0.25 s with `asyncio.timeout`. How many steps finish? Now give each step its *own* 0.25 s `wait_for` instead — what happens to the total time, and why is that the wrong design?

In [ ]:
# Solution — try it yourself first, then run this
finished = []

async def work(name):
    await asyncio.sleep(0.1)
    finished.append(name)

try:
    async with asyncio.timeout(0.25):
        for name in ["a", "b", "c"]:
            await work(name)
except TimeoutError:
    pass
print("total budget  -> finished:", finished)

finished.clear()
start = time.perf_counter()
for name in ["a", "b", "c"]:
    await asyncio.wait_for(work(name), timeout=0.25)      # each step "fits", so nothing ever times out
print(f"per-step only -> finished: {finished} in {time.perf_counter() - start:.2f} s (over the 0.25 s budget)")
assert finished == ["a", "b", "c"]

**3.** Write an async generator `batch_stream(agen, size)` that groups an async stream of tokens into lists of `size` (the last may be shorter) — the shape you need to flush partial output to a client every few tokens.

In [ ]:
# Solution — try it yourself first, then run this
async def batch_stream(agen, size):
    batch = []
    async for item in agen:
        batch.append(item.strip())
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch

batches = [b async for b in batch_stream(stream_tokens("one two three four five six seven", delay=0.005), size=3)]
print(batches)
assert batches == [["one", "two", "three"], ["four", "five", "six"], ["seven"]]

## Say it in an interview

- **Why async for LLM endpoints (tl08):** "The request is I/O-bound — the process waits on the provider. A sync handler ties up a worker for the whole generation, so concurrency equals worker count. An async handler yields at every `await`, so one process holds hundreds of in-flight requests and the real limit becomes the provider's rate limit."
- **The trap:** "One blocking call inside `async def` — the sync SDK, `requests`, `time.sleep`, a heavy parse — stalls every request on that loop. Use async clients or `asyncio.to_thread`."
- **Fan-out safely (ag32 / in29):** "`gather` with a `Semaphore` sized to my quota; `return_exceptions=True` or a `TaskGroup` depending on whether partial success is acceptable; and I tell the model which branches failed."
- **Timeouts (in26 / po17):** "Explicit timeout at every hop, one deadline for the whole request so steps fit inside what is left, fall back to a smaller model and label the answer degraded — never cache it."
- **Streaming (pd39):** "Improves time to first token, not total time or cost. Output guardrails and JSON validation can't see the whole answer until the end, and cancellation must propagate when the client disconnects."
- **CPU-bound:** "Async and threads don't parallelise pure-Python CPU work because of the GIL — use processes (or libraries that release the GIL); 3.13+ has an experimental free-threaded build."
- **Jupyter vs scripts:** "Scripts call `asyncio.run(main())` once; Jupyter already runs a loop, so you `await` at the top level and `asyncio.run` raises."

## Next

- Production depth: [async concurrency for LLM calls](../16_production/01_async_concurrency_for_llm_calls.ipynb) · [retries, backoff, rate limits and circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb) · [FastAPI serving and streaming](../16_production/04_fastapi_serving_and_streaming.ipynb) · [latency percentiles and cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb)
- [OpenAI API basics](../08_llm_apps/01_openai_api_basics.ipynb) — the sync client, streaming and parameters
- [15 · Testing and debugging](15_testing_and_debugging.ipynb)
- Theory: [python_basics course](../../python_basics/index.html) · [ai_system_design_concepts course](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)